# ECG-FM Embedding Playground
Extracts embeddings from MIMIC-IV ECG records using the ECG Foundation Model (Meta AI).

**Pipeline:**
1. Load config + model
2. Probe output shape to confirm `hidden_dim`
3. Run batched GPU extraction
4. Save embeddings to parquet

In [42]:
import pandas as pd
import os
import sys
import json
import importlib
from pathlib import Path
import numpy as np
import torch

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "configs" / "ecg_fm_playground_params.json"

In [43]:
import src.models.ecg_fm_playground as efp
importlib.reload(efp)

print("Module loaded:", efp.__file__)

Module loaded: /home/b5ng/private/ClinicalDigitalTwin/src/models/ecg_fm_playground.py


## 1. Load config and model

In [44]:
cfg = efp.load_config(CONFIG_PATH)
print(json.dumps(cfg, indent=2))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

{
  "paths": {
    "base_records_dir": "/home/b5ng/teams/b03/ecg/files"
  },
  "model": {
    "checkpoint_path": "src/models/ecgfm_pretrained.pt",
    "embedding_dim": 512
  }
}

Using device: cuda
GPU: NVIDIA GeForce GTX 1080 Ti
VRAM: 11.7 GB


In [45]:
REPO_ROOT / cfg["model"]["checkpoint_path"]

PosixPath('/home/b5ng/private/ClinicalDigitalTwin/src/models/ecgfm_pretrained.pt')

In [46]:
model = efp.load_ecgfm_model(REPO_ROOT / cfg["model"]["checkpoint_path"], device)
print("Model loaded successfully.")

Model loaded successfully.


## 2. Probe output shape

Run this **before** extraction to confirm `hidden_dim` and update `embedding_dim` in the config if needed.
- If `hidden_dim = 256` → `embedding_dim = 512` ✔️
- If `hidden_dim = 512` → `embedding_dim = 1024` → update the JSON

In [47]:
hidden_dim = efp.probe_model_output(model, device)

configured_dim = cfg["model"]["embedding_dim"]
actual_dim = hidden_dim * 2

Output keys: ['x', 'padding_mask', 'features_pen', 'features', 'prob_perplexity', 'code_perplexity', 'num_vars', 'temp']
features shape: (1, 312, 768)  →  (batch, seq_len, hidden_dim)
hidden_dim = 768
Concatenated embedding dim (2 segments) = 1536


## 3. Build subject/path list

Expects a DataFrame with at minimum: `subject_id`, `study_id`, `ecg_path`
where `ecg_path` is the WFDB base-path (no `.hea`/`.dat` extension).

In [48]:
# --- Load your ECG manifest here ---
# Example: load from MIMIC-IV-ECG record list
BASE_DIR = Path(cfg["paths"]["base_records_dir"])

# Uncomment and adjust to your manifest:
# record_df = pd.read_csv(BASE_DIR / "record_list.csv")
# record_df["ecg_path"] = record_df["path"].apply(lambda p: str(BASE_DIR / p))

# --- Quick smoke-test with a single record ---
# Replace with a real path from your MIMIC-IV-ECG mount:
test_path = str(BASE_DIR / "p1000" / "p10000032" / "s40689238" / "40689238")
record_df = pd.DataFrame({
    "subject_id": [10000032],
    "study_id": [40689238],
    "ecg_path": [test_path],
})

print(f"{len(record_df)} records to process")
record_df.head()

1 records to process


,subject_id,study_id,ecg_path
0,10000032,40689238,/home/b5ng/teams/b03/ecg/files/p1000/p10000032...


## 4. Run batched embedding extraction

In [50]:
embs = efp.extract_embeddings_batched(
    model=model,
    signal_paths=record_df["ecg_path"].tolist(),
    device=device,
    batch_size=32,
    n_channels=12,
    target_samples=5000,
    segment_split=True,   # two 2500-sample halves → concatenated dim
)

print(f"Embedding matrix shape: {embs.shape}")
print(f"  rows = records, cols = embedding dimensions")

  [1/1] batches processed
Embedding matrix shape: (1, 1536)
  rows = records, cols = embedding dimensions


## 5. Attach to DataFrame and save

In [51]:
emb_cols = [f"emb_{i}" for i in range(embs.shape[1])]
emb_df = pd.DataFrame(embs, columns=emb_cols)

result_df = pd.concat(
    [record_df.reset_index(drop=True), emb_df],
    axis=1
)

print(result_df.shape)
result_df.head()

(1, 1539)


,subject_id,study_id,ecg_path,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,emb_1526,emb_1527,emb_1528,emb_1529,emb_1530,emb_1531,emb_1532,emb_1533,emb_1534,emb_1535
0,10000032,40689238,/home/b5ng/teams/b03/ecg/files/p1000/p10000032...,-0.154856,-0.081678,-0.041643,13.509462,0.061587,-35.755161,0.031271,...,-2.592019,-0.158612,0.008387,-1.439813,-0.094804,-0.197356,-44.500568,0.118674,0.035714,5.788039


## 6. GPU cleanup

In [52]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory released.")

GPU memory released.
